In [ ]:
import equinox as eqx
import grain
import jax
from context_flux_no.data import SpatialDownsample, TheWellDataSource


jax.config.update("jax_default_device", jax.devices("gpu")[6])

In [6]:
source = TheWellDataSource(
    "data/datasets",
    "euler_multi_quadrants_periodicBC",
    window_size=21,
    exclude_field_names=["pressure"],
)

In [ ]:
dataset = (
    grain.MapDataset.source(source)
    .apply(SpatialDownsample(downsample=4))
    .shuffle(10)
    .to_iter_dataset()
)
performance_config = grain.experimental.pick_performance_config(
    ds=dataset, ram_budget_mb=8192, max_workers=8, max_buffer_size=None
)

prefetch_lazy_iter_ds = dataset.mp_prefetch(
    performance_config.multiprocessing_options,
).batch(batch_size=128)

In [9]:
iterator = iter(prefetch_lazy_iter_ds)
batch = next(iterator)

In [10]:
batch = next(iterator)

In [11]:
batch.shape

(128, 21, 4, 128, 128)

In [14]:
encoder = TRecViTEncoder(
    num_spatial_dims=2,
    in_channels=4,
    grid_size=(128, 128),
    patch_size=(16, 16),
    embedding_dim=128,
    depth=3,
    temporal_block_width=128,
    num_heads=8,
    mlp_hidden_dim=128,
    key=jax.random.key(0),
)


In [15]:
eqx.filter_vmap(encoder)(batch[:, :-1]).shape

(128, 20, 128, 64)

In [16]:
batch[:, :-1].shape

(128, 20, 4, 128, 128)